<a href="https://colab.research.google.com/github/KatherineElkins/programminghumanity2026/blob/main/pong_three_paradigms_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# One Pong Game, Three Programming Paradigms

The rules and starting conditions stay the same. Only the organization of the code changes.

1. **Procedural:** procedures change shared state.
2. **Object-oriented:** objects combine state and behavior.
3. **Functional:** pure functions transform an old state into a new state.

Each version includes a small change for you to make. Run the cells from top to bottom.


## The shared game

Both paddles automatically move toward the ball. The simulation records the ball's path, so it runs reliably in Colab without opening a separate game window.


In [ ]:
import copy
import matplotlib.pyplot as plt

WIDTH, HEIGHT = 100, 60
LEFT_X, RIGHT_X = 6, 94
PADDLE_HALF = 8
BASE_SPEED = 1.6
STEPS = 180

def starting_values():
    return {
        "ball_x": 50.0, "ball_y": 18.0, "vx": 2.2, "vy": 1.35,
        "left_y": 30.0, "right_y": 30.0,
        "left_score": 0, "right_score": 0,
    }

def show_game(history, title):
    xs = [frame["ball_x"] for frame in history]
    ys = [frame["ball_y"] for frame in history]
    final = history[-1]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.set_facecolor("#10212b")
    ax.plot(xs, ys, color="#ffd166", linewidth=2)
    ax.scatter(xs[0], ys[0], color="#06d6a0", s=70, label="start")
    ax.scatter(xs[-1], ys[-1], color="#ef476f", s=70, label="finish")
    ax.vlines(LEFT_X, final["left_y"]-PADDLE_HALF, final["left_y"]+PADDLE_HALF, color="white", linewidth=7)
    ax.vlines(RIGHT_X, final["right_y"]-PADDLE_HALF, final["right_y"]+PADDLE_HALF, color="white", linewidth=7)
    ax.set(xlim=(0, WIDTH), ylim=(0, HEIGHT), title=title,
           xlabel="court width", ylabel="court height")
    ax.legend(loc="upper center", ncol=2)
    plt.show()
    print(f"Final score: {final['left_score']}–{final['right_score']}")


## 1. Procedural Pong

The game is a dictionary of shared variables. Procedures receive that dictionary and change it **in place**. Read procedural_step as a recipe: move paddles, move ball, check walls, then check paddles and goals.


In [ ]:
def procedural_move_paddle(game, key, paddle_speed):
    difference = game["ball_y"] - game[key]
    game[key] += max(-paddle_speed, min(paddle_speed, difference))

def procedural_step(game, paddle_speed=BASE_SPEED):
    procedural_move_paddle(game, "left_y", paddle_speed)
    procedural_move_paddle(game, "right_y", paddle_speed)
    game["ball_x"] += game["vx"]
    game["ball_y"] += game["vy"]

    if game["ball_y"] <= 0 or game["ball_y"] >= HEIGHT:
        game["vy"] *= -1
        game["ball_y"] = max(0, min(HEIGHT, game["ball_y"]))

    hit_left = (game["vx"] < 0 and game["ball_x"] <= LEFT_X
                and abs(game["ball_y"] - game["left_y"]) <= PADDLE_HALF)
    hit_right = (game["vx"] > 0 and game["ball_x"] >= RIGHT_X
                 and abs(game["ball_y"] - game["right_y"]) <= PADDLE_HALF)
    if hit_left or hit_right:
        game["vx"] *= -1

    if game["ball_x"] < 0:
        game["right_score"] += 1
        game.update({"ball_x": 50.0, "ball_y": 30.0, "vx": 2.2, "vy": 1.35})
    elif game["ball_x"] > WIDTH:
        game["left_score"] += 1
        game.update({"ball_x": 50.0, "ball_y": 30.0, "vx": -2.2, "vy": 1.35})

def run_procedural(paddle_speed=BASE_SPEED, steps=STEPS):
    game = starting_values()
    history = [copy.deepcopy(game)]
    for _ in range(steps):
        procedural_step(game, paddle_speed)
        history.append(copy.deepcopy(game))
    return history

procedural_history = run_procedural()
show_game(procedural_history, "Procedural Pong")


### Modify the procedural version

Change PROCEDURAL_SPEED. Predict the result before running it.

**Question:** Which shared values and procedures produced the change?


In [ ]:
PROCEDURAL_SPEED = 0.7  # TODO: try 0.4, 1.0, or 3.0
procedural_modified = run_procedural(paddle_speed=PROCEDURAL_SPEED)
show_game(procedural_modified, f"Procedural Pong: paddle speed = {PROCEDURAL_SPEED}")


## 2. Object-Oriented Pong

Each entity owns its state and relevant behavior. A Paddle knows how to move, a Ball knows how to move and bounce, and a PongGame coordinates them.


In [ ]:
class Paddle:
    def __init__(self, x, y=30.0, speed=BASE_SPEED):
        self.x, self.y, self.speed = x, y, speed

    def move_toward(self, target_y):
        difference = target_y - self.y
        self.y += max(-self.speed, min(self.speed, difference))

    def can_hit(self, ball_y):
        return abs(ball_y - self.y) <= PADDLE_HALF

class Ball:
    def __init__(self):
        self.x, self.y = 50.0, 18.0
        self.vx, self.vy = 2.2, 1.35

    def move(self):
        self.x += self.vx
        self.y += self.vy

    def bounce_horizontal(self):
        self.vx *= -1

    def bounce_vertical(self):
        self.vy *= -1

    def reset(self, direction):
        self.x, self.y = 50.0, 30.0
        self.vx, self.vy = 2.2 * direction, 1.35

class PongGame:
    def __init__(self, paddle_speed=BASE_SPEED):
        self.left = Paddle(LEFT_X, speed=paddle_speed)
        self.right = Paddle(RIGHT_X, speed=paddle_speed)
        self.ball = Ball()
        self.left_score = self.right_score = 0

    def snapshot(self):
        return {
            "ball_x": self.ball.x, "ball_y": self.ball.y,
            "vx": self.ball.vx, "vy": self.ball.vy,
            "left_y": self.left.y, "right_y": self.right.y,
            "left_score": self.left_score, "right_score": self.right_score,
        }

    def step(self):
        self.left.move_toward(self.ball.y)
        self.right.move_toward(self.ball.y)
        self.ball.move()
        if self.ball.y <= 0 or self.ball.y >= HEIGHT:
            self.ball.bounce_vertical()
            self.ball.y = max(0, min(HEIGHT, self.ball.y))

        hit_left = (self.ball.vx < 0 and self.ball.x <= LEFT_X
                    and self.left.can_hit(self.ball.y))
        hit_right = (self.ball.vx > 0 and self.ball.x >= RIGHT_X
                     and self.right.can_hit(self.ball.y))
        if hit_left or hit_right:
            self.ball.bounce_horizontal()

        if self.ball.x < 0:
            self.right_score += 1
            self.ball.reset(1)
        elif self.ball.x > WIDTH:
            self.left_score += 1
            self.ball.reset(-1)

    def run(self, steps=STEPS):
        history = [self.snapshot()]
        for _ in range(steps):
            self.step()
            history.append(self.snapshot())
        return history

oop_history = PongGame().run()
show_game(oop_history, "Object-Oriented Pong")


### Modify the object-oriented version

Add behavior to one object. Change FastBall.bounce_horizontal so the ball moves **10% faster** after every paddle hit.

**Question:** Why is the acceleration rule naturally located inside Ball rather than PongGame?


In [ ]:
class FastBall(Ball):
    def bounce_horizontal(self):
        self.vx *= -1  # TODO: multiply by 1.1 as well

fast_game = PongGame()
fast_game.ball = FastBall()
oop_modified = fast_game.run()
show_game(oop_modified, "Object-Oriented Pong: FastBall")


## 3. Functional Pong

State is never changed in place. Each function receives values and returns new values. functional_step constructs and returns a **new state**.


In [ ]:
def functional_move_paddle(paddle_y, ball_y, paddle_speed):
    difference = ball_y - paddle_y
    return paddle_y + max(-paddle_speed, min(paddle_speed, difference))

def functional_step(old, paddle_speed=BASE_SPEED):
    new = old.copy()
    new["left_y"] = functional_move_paddle(old["left_y"], old["ball_y"], paddle_speed)
    new["right_y"] = functional_move_paddle(old["right_y"], old["ball_y"], paddle_speed)
    new["ball_x"] = old["ball_x"] + old["vx"]
    new["ball_y"] = old["ball_y"] + old["vy"]

    if new["ball_y"] <= 0 or new["ball_y"] >= HEIGHT:
        new["vy"] = -old["vy"]
        new["ball_y"] = max(0, min(HEIGHT, new["ball_y"]))

    hit_left = (old["vx"] < 0 and new["ball_x"] <= LEFT_X
                and abs(new["ball_y"] - new["left_y"]) <= PADDLE_HALF)
    hit_right = (old["vx"] > 0 and new["ball_x"] >= RIGHT_X
                 and abs(new["ball_y"] - new["right_y"]) <= PADDLE_HALF)
    if hit_left or hit_right:
        new["vx"] = -old["vx"]

    if new["ball_x"] < 0:
        new.update({"ball_x": 50.0, "ball_y": 30.0, "vx": 2.2, "vy": 1.35,
                    "right_score": old["right_score"] + 1})
    elif new["ball_x"] > WIDTH:
        new.update({"ball_x": 50.0, "ball_y": 30.0, "vx": -2.2, "vy": 1.35,
                    "left_score": old["left_score"] + 1})
    return new

def run_functional(step_function=functional_step, paddle_speed=BASE_SPEED, steps=STEPS):
    history = [starting_values()]
    for _ in range(steps):
        history.append(step_function(history[-1], paddle_speed))
    return history

functional_history = run_functional()
show_game(functional_history, "Functional Pong")


### Modify the functional version

Add **spin** based on where the ball strikes a paddle. A hit near the paddle's center should add little spin; a hit near an edge should change the vertical velocity more.

Complete add_spin so it returns a new vertical velocity without changing any outside value.

**Question:** Why is add_spin a pure function? What information would it need as an input if we wanted a more complex spin rule?


In [ ]:
def add_spin(vertical_velocity, ball_y, paddle_y):
    impact_offset = (ball_y - paddle_y) / PADDLE_HALF
    return vertical_velocity  # TODO: add impact_offset * 0.8

def functional_spin_step(old, paddle_speed=BASE_SPEED):
    new = functional_step(old, paddle_speed)
    hit_left = (old["vx"] < 0 and new["ball_x"] <= LEFT_X
                and abs(new["ball_y"] - new["left_y"]) <= PADDLE_HALF)
    hit_right = (old["vx"] > 0 and new["ball_x"] >= RIGHT_X
                 and abs(new["ball_y"] - new["right_y"]) <= PADDLE_HALF)
    if hit_left:
        new["vy"] = add_spin(new["vy"], new["ball_y"], new["left_y"])
    elif hit_right:
        new["vy"] = add_spin(new["vy"], new["ball_y"], new["right_y"])
    return new

functional_modified = run_functional(step_function=functional_spin_step)
show_game(functional_modified, "Functional Pong: add spin")


## Do the baseline programs behave alike?

Different organizations of code can describe the same computation. This cell compares every recorded baseline state.


In [ ]:
def compact(history):
    keys = ("ball_x", "ball_y", "vx", "vy", "left_y", "right_y",
            "left_score", "right_score")
    return [tuple(round(frame[k], 8) if isinstance(frame[k], float) else frame[k]
                  for k in keys) for frame in history]

assert compact(procedural_history) == compact(oop_history) == compact(functional_history)
print("✓ All three baseline implementations produced the same 180-step game.")


## Compare the ways of thinking

| Paradigm | Where is state? | How does change happen? | Natural question |
|---|---|---|---|
| Procedural | Shared dictionary | Procedures mutate it in sequence | What step happens next? |
| Object-oriented | Inside interacting objects | Methods change each object's state | Which object should own this behavior? |
| Functional | In explicit state snapshots | Functions return transformed values | What new state follows from this state? |

**Discuss:**

1. In which version was your modification most local?
2. Which version made the flow of time easiest to follow?
3. Which made unintended side effects easiest to avoid?
4. Did the paradigm merely change syntax, or did it change how you imagined the game?
